In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..","python"]))

import numpy as np
from gridr.core.grid.grid_resampling import array_grid_resampling
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook # enables plot interface in J notebook
    output_notebook()

image = mandrill[0]
grid_dtype = np.float64
data_in_dtype = np.float64
data_out_dtype = np.float64
array_in = image.astype(data_in_dtype)

# Build the identity transformation grid used as the starting point in
# several tutorials.
x = np.arange(0, image.shape[0], dtype=grid_dtype)
y = np.arange(0, image.shape[1], dtype=grid_dtype)
xx, yy = np.meshgrid(x, y)

# Pipeline integration: ``standalone=False``

If you're integrating GridR into a larger processing pipeline where you already handle preprocessing yourself, you can bypass the standalone preparation step entirely. This page shows when and how to do that.

**What you'll learn**

- When to set ``standalone=False``
- Adjusting the input coordinate origin with ``array_in_origin``
- What preprocessing responsibilities you take on

## Advanced Usage: Pipeline Integration (standalone=False)

### Adjusting the Origin of Input Coordinates

The `array_grid_resampling` function includes the parameter `array_in_origin` (default: (0., 0.)) that allows shifting the origin of the input coordinate system.

From an implementation standpoint, this shift is applied directly to the computed target grid coordinates.

This feature is useful in several scenarios:

- Handling different grid conventions without modifying the grid itself. As a reminder, GridR assumes that integer coordinates refer to pixel centers. In other systems, a pixel may be treated as an area, with its center located relatively at (0.5, 0.5).

- Working with subregions of a larger input array. In this case, the shift can represent the position of the subregion within the full array’s coordinate system.

As an example we will take a previous example using an idendity transformation.

In [ ]:
# create identity grid
if image.ndim == 2:
    x = np.arange(0, image.shape[0], dtype=grid_dtype)
    y = np.arange(0, image.shape[1], dtype=grid_dtype)
xx, yy = np.meshgrid(x, y)

win_center = np.array((58, 175)) # left eye
win_shape =  np.array((100, 100))
win = np.array((win_center - win_shape//2, win_center - win_shape//2 + win_shape - 1)).T

# Call it without specifying array_in_origin
array_out_origin_0_0, _ = array_grid_resampling(
        interp="cubic",
        array_in=array_in,
        grid_row=yy,
        grid_col=xx,
        grid_resolution=(1,1),
        array_out=None,
        win=win,
        array_in_mask=None,
        grid_mask=None,
        array_out_mask=None,
        nodata_out=None,
        )

# Call it with specifying array_in_origin and passing standalone=False
array_out_origin_30_m10, _ = array_grid_resampling(
        interp="cubic",
        array_in=array_in,
        array_in_origin=(30, -10), # <= shift +30 on rows and -10 on columns
        grid_row=yy,
        grid_col=xx,
        grid_resolution=(1,1),
        array_out=None,
        win=win,
        array_in_mask=None,
        grid_mask=None,
        array_out_mask=None,
        nodata_out=None,
        standalone=False,
        )

In [ ]:
plot_im({'no shift':array_out_origin_0_0, 'shift':array_out_origin_30_m10}, prefix='008-array-origin')

As you can observe, the coordinate shift is reflected in the bottom image.